### This JOURNALS notebook 

-- Loads journal data into the database  
-- Finds the ISSN of Domingo's Incites journals  
-- EXamines the "completeness" of the Incites journals 


In [1]:
import duckdb
import pandas as pd
from pathlib import Path
import diskcache
from itertools import chain

from utils.pandas_setup import pandas_setup
pandas_setup()

import pyalex
from pyalex import Works, Authors, Sources, Institutions, Topics, Publishers, Funders
pyalex.config.email = "Lawrence.Cram@anu.edu.au"
pyalex.config.max_retries = 0
pyalex.config.retry_backoff_factor = 0.1
pyalex.config.retry_http_codes = [429, 500, 503]

MY_DATA_PATH = Path('../DATA/')
MY_DATABASE_FILE = Path(MY_DATA_PATH / 'econ.duckdb')
MY_CACHE_FILE = Path('/home/lc/m/.cache/econommicsbusiness/cache.db')
DATAFILES_PATH = Path('../DATAFILES')

In [2]:
class SetUp:

    def __init__(self):
        self._setup_db()
        self._setup_cache()
        return
    
    def _setup_db(self):
        self.db = duckdb.connect(MY_DATABASE_FILE)
        self.db.sql("SHOW ALL TABLES").show()
        return
    
    def _setup_cache(self):
        self.cache = diskcache.Cache(MY_CACHE_FILE, size_limit=4_000_000_000)
        print(f'{self.cache.check() = }')
        print(f'{self.cache.volume() = }')
        return
    
    def name_of_global_obj(self, obj=None):
        for objname, oid in globals().items():
            if oid is obj:
                return objname

In [3]:
class JournalETL(SetUp):

    def __init__(self):
        super().__init__()
        return
    
    def extract_incites_journals(self):
        # Extract Domingo's inCites journal table 
        self.incites = pd.read_excel(DATAFILES_PATH / 'data_economics.xlsx')
        self.incites['issn'] = pd.NA
        self.incites['eissn'] = pd.NA
        print(f'{self.incites.shape = }\n{self.incites.head()}')
        return
    
    def transform_incites_journals(self):
        # insert ISSN and eISSN into inCites journals from ESI+emerging journals lists
        esi_journals = pd.read_csv(MY_DATA_PATH / 'JOURNAL_DATA/Essential Science Indicators_emerging.csv')
        esi_journals.columns = [c.replace(' ', '_') for c in esi_journals.columns]
        print(f'{esi_journals.shape = }\n{esi_journals.head()}')
        for row in esi_journals.itertuples():
            self.incites.loc[self.incites.eco_incites == row.Journal_title, ['issn', 'eissn']] = [[row.ISSN, row.eISSN]]
        print(f'{self.incites.shape = }\n{self.incites.head()}') 
        print(f'{self.incites[self.incites.issn.isna()].head()}')        
        return
    
    def match_incites_oa(self):

        hold = []
        for row in self.incites.itertuples():
            issn = row.issn if isinstance(row.issn, str) else row.eissn
            if not isinstance(issn, str):
                print(f'INCITES does not have ISSN FOR {row.eco_incites = } {row.pub = }')
                continue
            reader = f'Sources().filter(issn="{issn}")'
            if oa := self.read_openalex(reader=reader):
                # print(f'{oa = }')
                hold.extend(oa)
            else:
                print(f'OpenAlex does not have ISSN FOR {issn = } {row.eco_incites = } {row.pub = }')
        # print(f'{type(hold) = }\n{hold = }')
        df = pd.DataFrame.from_dict(hold)
        print(f'{df.shape = }\n{df.head()}')
        df.to_excel(MY_DATA_PATH / 'sources_oa_incites.xlsx')
        self.db.sql("CREATE OR REPLACE TABLE sources_oa_incites AS SELECT * FROM df")
        self.db.sql("SELECT * FROM sources_oa_incites").show()
        return
    
    def read_openalex(self, reader=None):
        # print(f'{eval(reader) = }')
        if result := self.cache.get(reader):
            # print(f'READ FROM CACHE {reader = } {result = }')
            return result
        try:
            result = list(chain(*eval(reader).paginate(per_page=200)))
            # print(f'{result = }')
        except Exception as e:
            print(f'FAILED TO READ cache or OpenAlex API with {reader = }')
            print(f'{e = }')
            return
        self.cache[reader] = result
        # print(f'READ FROM API and loaded cache {reader = } {result = }')
        return result


In [4]:
def main():

    jetl = JournalETL()
    jetl.extract_incites_journals()
    jetl.transform_incites_journals()
    jetl.match_incites_oa() 

In [5]:
if __name__ == "__main__":
    main()
    print("DONE!")


┌──────────┬─────────┬─────────┬──────────────┬──────────────┬───────────┐
│ database │ schema  │  name   │ column_names │ column_types │ temporary │
│ varchar  │ varchar │ varchar │  varchar[]   │  varchar[]   │  boolean  │
├──────────┴─────────┴─────────┴──────────────┴──────────────┴───────────┤
│                                 0 rows                                 │
└────────────────────────────────────────────────────────────────────────┘

self.cache.check() = []
self.cache.volume() = 12992512
self.incites.shape = (579, 6)
                                   eco_incites  acr  pub  journal_score  issn eissn
0                    APPLIED ECONOMICS LETTERS  AAA  435              1  <NA>  <NA>
1                                ENERGY POLICY  AAB  662              1  <NA>  <NA>
2                            APPLIED ECONOMICS  AAC  374              1  <NA>  <NA>
3                            ECONOMICS LETTERS  AAD  438              1  <NA>  <NA>
4  JOURNAL OF ECONOMIC BEHAVIOR & ORGANIZATI